In [1]:
import sys, os, json
import pandas as pd
from pathlib import Path

sys.path.insert(1,os.path.abspath('../..'))
from tools.battle import *

log_dir = Path("../../data/replays/gen9-randombattle")
log_dir2 = Path("../../data/replays/gen9-randombattle_2")
log_dir3 = Path("../../data/replays/gen9-randombattle_3")

# helpful to not have to use the full filename
def pull_by_num(num, ns='', parse=False):
    return Battle(f'../../data/replays/gen9-randombattle{ns}/gen9randombattle-{num}.json', parse)

In [5]:
error_ids = [] # battles that have trouble loading

rows=[] # for df
col_names = ['id','w_p1','d_elo','n_turns','time'] # for df

# adv{i}{j} = advantage(team_1_mon_{i}, team_2_mon_{j})
for i in range(6):
    for j in range(6):
        col_names.append(f"adv{i+1}{j+1}")

# easier to call on multiple directories
def foo(file):
    try : 
        bat = Battle(file)
        # throw out battles with custom rules and those that last less than 60 seconds
        if not (bat.custom_ruleQ) and not (bat.end_time - bat.start_time < 60): 
            # with file.open() as battle_json:
            #     battle_data = json.load(battle_json)
            
            rows.append([
                bat.id.removeprefix('gen9randombattle-'), 
                int(bat.winner.name == bat.p1.name), # w_p1
                bat.p1.elo0-bat.p2.elo0, # d_elo
                len(bat.STATES), # n_turns
                bat.end_time-bat.start_time # time
                ])
            
            team1 = [FullPokemon(bat.teams_full[0][mon]) for mon in bat.teams_full[0].keys()]
            team2 = [FullPokemon(bat.teams_full[1][mon]) for mon in bat.teams_full[1].keys()]
            
            for i in range(6):
                for j in range(6):
                    rows[-1].append(FullPokemon.advantage(team1[i],team2[j]))
    except : 
        print(f"error with {file.stem}")
        error_ids.append(file.stem)


for file in log_dir.iterdir():
    if file.name.endswith('.json'):
        foo(file)
for file in log_dir2.iterdir():
    if file.name.endswith('.json'):
        foo(file)
for file in log_dir3.iterdir():
    if file.name.endswith('.json'):
        foo(file)

# should have ~12k+ rows, 41 columns
df = pd.DataFrame(rows,columns=col_names)
df.drop_duplicates(subset=['id'], ignore_index=True, inplace=True) # should be redundant, but good to be sure
df 

,id,w_p1,d_elo,n_turns,time,adv11,adv12,adv13,adv14,adv15,...,adv53,adv54,adv55,adv56,adv61,adv62,adv63,adv64,adv65,adv66
0,2631906096,0,-5,33,598,1.147629,1.123535,1.705675,0.858827,0.693693,...,0.912017,0.872012,0.853608,-0.258471,1.121944,1.654554,1.005346,-0.789109,0.170756,-1.030727
1,2631763570,0,10,23,167,-0.826425,-0.051193,-1.227396,0.514911,-0.545356,...,-0.714277,-0.365187,-1.164154,-0.698862,0.791319,0.585443,0.397549,0.347256,1.749648,1.177513
2,2631369343,1,-69,23,275,0.328214,1.037827,1.262965,-0.386739,0.718781,...,0.504586,1.162076,-0.947110,1.119793,1.005441,-1.007017,-0.120664,-0.500574,0.305740,-0.185950
3,2631529004,1,17,10,123,-0.173456,-0.971856,0.274374,-0.283311,0.510040,...,0.529256,0.197303,-1.129192,-0.442040,1.063704,0.927443,0.942618,-0.156272,1.043459,-0.128803
4,2631993792,0,58,24,301,-0.290351,0.339730,0.159254,-0.811030,-0.439897,...,-1.172650,0.275969,-0.469168,1.130988,1.033497,-0.499937,-0.197355,1.482827,0.203640,1.341519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12264,2642081603,0,-68,32,436,0.876270,-0.416519,-0.675781,0.416327,-0.111300,...,0.591170,0.968801,0.289080,-0.129635,-0.361229,1.023727,-0.644765,-0.015020,0.510183,-0.537115
12265,2641877736,0,-8,31,239,-0.297292,-0.420925,-0.330454,-0.771071,-0.795918,...,-0.256885,-0.692387,-0.257532,-0.350974,-0.611551,-0.373638,-0.207030,-0.613313,-0.406860,-0.158083
12266,2642095210,0,-49,23,187,-0.521765,0.873065,-1.419255,0.298763,1.672853,...,1.019051,0.659662,-0.257854,-0.984054,-0.366654,-0.797029,1.445350,0.551774,-0.422699,0.301263
12267,2642125538,0,4,29,186,1.048859,-0.770079,0.835945,-0.245636,0.234887,...,-0.673489,-0.273518,0.994666,-0.405209,0.967479,-0.903486,-0.573505,-0.203632,0.619339,-0.474474


# EDA

(Things you can run for some basic plots)

In [ ]:
import ydata_profiling as ydata
profile = ydata.ProfileReport(df, explorative=True)

In [ ]:
import seaborn as sns
sns.pairplot(df, x_vars=['d_elo', 'n_turns', 'time'], y_vars=['w_p1'])

# Modeling

In [44]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import( 
    RandomForestClassifier, 
    ExtraTreesClassifier,
    AdaBoostClassifier
)
from sklearn.model_selection import(
    GridSearchCV, 
    train_test_split, 
    cross_val_score, 
    cross_validate
)
from xgboost import XGBClassifier

# grab first few columns
X0 = df[['d_elo','n_turns','time']].copy() # include this .copy() or face the ire of pandas
y0 = df.w_p1

# for a given Mon {i}, compute its 'total advantage vs team 2' 
# (could average, but this is taken care of by StandardScaler later)
for i in range(6):
    X0[f'A{i+1}'] = df[df.columns[5+6*i : 5+6*(i+1)]].sum(axis=1)
print(X0.head())
print()
print()
print(y0.head())

   d_elo  n_turns  time        A1        A2        A3        A4        A5  \
0     -5       33   598  5.844212  1.129274 -1.850993 -1.124150  0.573367   
1     10       23   167 -3.433173  3.360181 -0.454044  0.693177 -1.282305   
2    -69       23   275  3.781412  1.194816  0.884586  3.361011  3.936347   
3     17       10   123  0.046445  0.695683  1.187595 -2.032783  1.296550   
4     58       24   301 -1.836523  1.784906  0.811944  0.713204  1.071657   

         A6  
0  2.132764  
1  5.048728  
2 -0.503025  
3  3.692149  
4  3.364192  


0    0
1    0
2    1
3    1
4    0
Name: w_p1, dtype: int64


### "SideQuest:"
I wanted to be extra sure that below when I use `scalar.fit_transform(X)` that each column gets transformed properly

In [29]:
col = 'd_elo'
scaler = StandardScaler()
scaler.fit(X0[[col]])
print(f"mean: {scaler.mean_}, std: {scaler.scale_}")
print()
print(f"transformed {[col]}:\n", scaler.transform(X0[[col]]))

mean: [-13.03472166], std: [57.61981829]

transformed ['d_elo']:
 [[ 0.13944372]
 [ 0.39977081]
 [-0.97128523]
 ...
 [-0.62418243]
 [ 0.29563998]
 [-0.41592076]]


In [30]:
col = 'n_turns'
scaler = StandardScaler()
scaler.fit(X0[[col]])
print(f"mean: {scaler.mean_}, std: {scaler.scale_}")
print()
print(f"transformed {[col]}:\n", scaler.transform(X0[[col]]))

mean: [25.56394164], std: [11.31957512]

transformed ['n_turns']:
 [[ 0.65692027]
 [-0.22650511]
 [-0.22650511]
 ...
 [-0.22650511]
 [ 0.30355012]
 [ 0.74526281]]


In [32]:
scaler = StandardScaler()
X = pd.DataFrame(
    scaler.fit_transform(X0),
    columns=X0.columns,
    index=X0.index
)
X.head()

,d_elo,n_turns,time,A1,A2,A3,A4,A5,A6
0,0.139444,0.656920,1.730388,1.351059,0.651922,-0.717640,-0.429107,0.321864,1.009825
1,0.399771,-0.226505,-0.889161,-2.036219,0.894669,0.368514,0.543812,-0.146766,1.094410
2,-0.971285,-0.226505,-0.232754,0.724133,-0.013846,0.741689,0.828237,0.274772,0.397054
3,0.521257,-1.374958,-1.156586,0.209621,0.506967,0.617313,-0.458304,0.081017,2.097598
4,1.232818,-0.138163,-0.074730,-0.652715,0.810464,0.107595,0.337650,-0.418742,1.520635


In [ ]:
# Looks good! 
# /SideQuest

## Back to work

In [10]:
scaler = StandardScaler()

X = pd.DataFrame(
    scaler.fit_transform(X0),
    columns=X0.columns,
    index=X0.index
)

# From what I understand about SVC, it prefers to have classes {-1,1} vs {0,1}
y = y0.replace(0, -1)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    stratify = y, 
    random_state = 502
)

### SVC

In [29]:
# Trying out some SVC params (took -several- minutes to run, even with all cpu cores)

# linear
param_grid = [{
    'kernel': ['linear'], 
    'C': [0.1, 1, 10, 100]
}]
     
grid = GridSearchCV(SVC(), param_grid, cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)
grid.fit(X_train, y_train)
print("========== linear kernel ==========")
print(f"best params: {grid.best_params_}")
print(f"best (acc) score: {grid.best_score_}")
print()
print("mean scores over grid:")
print(grid.cv_results_['mean_test_score'])

========== linear kernel ==========
best params: {'C': 0.1, 'kernel': 'linear'}
best (acc) score: 0.5223345062938385

mean scores over grid:
[0.52233451 0.52233451 0.52233451 0.52233451]


In [27]:
# RBF
param_grid = [{
    'kernel': ['rbf'], 
    'C': [0.1, 1, 10, 100], 
    'gamma': [0.01, 0.1, 1, 'scale']
}]
     
grid = GridSearchCV(SVC(), param_grid, cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)
grid.fit(X_train, y_train)
print("========== RBF kernel ==========")
print(f"best params: {grid.best_params_}")
print(f"best (acc) score: {grid.best_score_}")
print()
print("mean scores over grid:")
print(grid.cv_results_['mean_test_score'])

==== RBF kernel ====
best params: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
best (acc) score: 0.5333109959143186

mean scores over grid:
[0.52233451 0.52592152 0.52233451 0.52755172 0.53016036 0.52472439
 0.51276893 0.52374666 0.533311   0.52059414 0.51037716 0.52015906
 0.52646465 0.51494326 0.5108117  0.51679061]


In [ ]:
# this still ran after 10-15 mins; pausing for now
param_grid = [{
    'kernel': ['poly'], 
    'C': [0.1, 1, 10], 
    'degree': [2, 3], 
    'gamma': [0.1, 1, 'scale']
}]
     
# grid = GridSearchCV(SVC(), param_grid, cv=5, 
#     n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
# )
# grid.fit(X_train, y_train)
# print("Poly kernel:")
# print(grid.best_params_, grid.best_score_)

### RandomForest

In [28]:
param_grid = [{
    'n_estimators': [50,100,200], 
    'max_depth': [2,3,5], 
    'criterion': ["gini","entropy","log_loss"]
}]

gridRF = GridSearchCV(
    RandomForestClassifier(), 
    param_grid, 
    cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)
gridRF.fit(X_train, y_train)
print("========== RF Classifier ==========")
print(f"best params: {gridRF.best_params_}")
print(f"best (acc) score: {gridRF.best_score_}")
print()
print("mean scores over grid:")
print(gridRF.cv_results_['mean_test_score'])

==== RF Classifier ====
best params: {'criterion': 'entropy', 'max_depth': 5, 'n_estimators': 100}
best (acc) score: 0.5348328531280259

mean scores over grid:
[0.52798621 0.5278781  0.52863856 0.52842117 0.52798715 0.53102975
 0.53418097 0.53363726 0.53135548 0.52353116 0.5274432  0.52787798
 0.53211641 0.52940002 0.52744314 0.53385471 0.53483285 0.53211635
 0.52439955 0.52592194 0.52885642 0.53211611 0.52766053 0.5309207
 0.53385518 0.53037757 0.53385536]


### ExtraTreesClassifiers

In [30]:
param_grid = [{
    'n_estimators': [25, 50, 100],
    'criterion': ["gini","entropy","log_loss"],
    'max_depth': [3, 5, 7]
}]

grid = GridSearchCV(ExtraTreesClassifier(), param_grid, cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)
grid.fit(X_train, y_train)
print("========== ET Classifier ==========")
print(f"best params: {grid.best_params_}")
print(f"best (acc) score: {grid.best_score_}")
print()
print("mean scores over grid:")
print(grid.cv_results_['mean_test_score'])

========== ET Classifier ==========
best params: {'criterion': 'entropy', 'max_depth': 7, 'n_estimators': 50}
best (acc) score: 0.5324421982381976

mean scores over grid:
[0.52287787 0.52222581 0.52287793 0.52624666 0.52733368 0.52646459
 0.53244149 0.53146441 0.52994243 0.52157364 0.52276917 0.52363885
 0.52744226 0.52820342 0.52820348 0.53037787 0.5324422  0.53005137
 0.52516012 0.52266048 0.52255172 0.5273338  0.52842064 0.52689896
 0.53005196 0.53081159 0.52874678]


### AdaBoost

In [40]:
param_grid = [{
    'estimator': [LogisticRegression(), DecisionTreeClassifier()], 
    'n_estimators': [25, 50, 100],
    'learning_rate': [0.5, 1, 2, 4]
}]

grid = GridSearchCV(AdaBoostClassifier(), param_grid, cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)
grid.fit(X_train, y_train)
print("========== AdaBoost Classifier ==========")
print(f"best params: {grid.best_params_}")
print(f"best (acc) score: {grid.best_score_}")
print()
print("mean scores over grid:")
print(grid.cv_results_['mean_test_score'])

========== AdaBoost Classifier ==========
best params: {'estimator': LogisticRegression(), 'learning_rate': 2, 'n_estimators': 50}
best (acc) score: 0.5280949035259666

mean scores over grid:
[0.5271157  0.5271157  0.5271157  0.52754965 0.52754965 0.52754965
 0.52244273 0.5280949  0.5280949  0.52233451 0.47766549 0.47766549
 0.50125145 0.5011424  0.50168617 0.49962101 0.50146866 0.50092512
 0.50635997 0.50233828 0.50016479 0.49853394 0.50222947 0.49983817]


### GradientBoost

In [42]:
param_grid = [{
    'estimator': [LogisticRegression(), DecisionTreeClassifier()], 
    'n_estimators': [25, 50, 100],
    'learning_rate': [0.5, 1, 2, 4]
}]

grid = GridSearchCV(AdaBoostClassifier(), param_grid, cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)
grid.fit(X_train, y_train)
print("========== GradientBoost Classifier ==========")
print(f"best params: {grid.best_params_}")
print(f"best (acc) score: {grid.best_score_}")
print()
print("mean scores over grid:")
print(grid.cv_results_['mean_test_score'])

========== GradientBoost Classifier ==========
best params: {'estimator': LogisticRegression(), 'learning_rate': 2, 'n_estimators': 50}
best (acc) score: 0.5280949035259666

mean scores over grid:
[0.5271157  0.5271157  0.5271157  0.52754965 0.52754965 0.52754965
 0.52244273 0.5280949  0.5280949  0.52233451 0.47766549 0.47766549
 0.49875156 0.50190256 0.50190297 0.5034243  0.50277195 0.50038058
 0.49712119 0.49962095 0.49972994 0.50364169 0.50081613 0.5002726 ]


### XGBoost

In [48]:
param_grid = [{
    'booster': ['gbtree', 'gblinear'], 
    'eta': [0.1, 0.3, 0.7, 1.0]
    # 'gamma': [0.0, 0.25, 0.5]
}]

grid = GridSearchCV(XGBClassifier(), param_grid, cv=5, 
    n_jobs=-1 #NOTE:`n_jobs=-1` will use all available cpu cores; melt cpu at own risk
)

grid.fit(X_train, y_train.replace(-1,0)) # seems XGBoost wants {0,1}
print("========== XGBoost Classifier ==========")
print(f"best params: {grid.best_params_}")
print(f"best (acc) score: {grid.best_score_}")
print()
print("mean scores over grid:")
print(grid.cv_results_['mean_test_score'])

========== XGBoost Classifier ==========
best params: {'booster': 'gblinear', 'eta': 0.1}
best (acc) score: 0.5238547693833692

mean scores over grid:
[0.51505308 0.50548957 0.50212089 0.50103276 0.52385477 0.52363732
 0.52363732 0.52363732]


### (Pulled some stuff from `sklearn` docs)

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from sklearn.datasets import make_hastie_10_2
from sklearn.metrics import accuracy_score, make_scorer
from sklearn.model_selection import GridSearchCV
scoring = {"AUC": "roc_auc", "accuracy": "accuracy", "f1": "f1"}